# Thermal line segments on colored maps
During an ampacity simulation the lines that are analysed on their ampacity are split into
thermal line segments, each of which keeps its own thermal state. The simulation writes these
segments to ``thermal_line_segments.csv`` and the resulting segment temperatures to
``line_segment_res.csv``.
With this notebook you can plot the formed segments on the grid as well as the segment
temperature for a specific timestamp, similar to ``plotting_utilities_colormap_lines.ipynb``.


In [1]:
# Some jupyter notebook magic to reload modules automatically when they change
# not necessary for this specific notebook but useful in general
%load_ext autoreload
%autoreload 2
# Gives you high resolution images within the notebook
%config InlineBackend.figure_format = 'retina'


## Load Data


In [2]:
from definitions import ROOT_DIR
import os
# The PSDM specific input models can be imported from the pypsdm.models.input and pypsdm.models.result. The `GridWithResults` container is located in pypsdm.models.gwr
from pypsdm.models.gwr import GridWithResults
grid_path = os.path.join(ROOT_DIR, "tests", "resources", "ampacity", "input")
result_path = os.path.join(ROOT_DIR, "tests", "resources", "ampacity", "results")
# IO data models in general have a from_csv method to parse psdm files
gwr = GridWithResults.from_csv(grid_path, result_path)


2026-09-27 16:22:16.386 | INFO     | pypsdm.models.gwr:from_csv:293 - Reading grid from /home/smdafeis/github/pypsdm/tests/resources/ampacity/input
2026-09-27 16:22:16.427 | DEBUG    | pypsdm.models.primary_data:from_csv:273 - No primary data in path /home/smdafeis/github/pypsdm/tests/resources/ampacity/input
2026-09-27 16:22:16.428 | INFO     | pypsdm.models.gwr:from_csv:305 - Reading results from /home/smdafeis/github/pypsdm/tests/resources/ampacity/results


## Get the Thermal Line Segments
The thermal line segments created during the simulation can be read directly from the
result data with ``pandas``:


In [3]:
# NBVAL_CHECK_OUTPUT
import pandas as pd
segments = pd.read_csv(os.path.join(result_path, "thermal_line_segments.csv"))
segments


,segmentUuid,lineUuid,startX,startY,endX,endY,limitTemperature
0,f860a372-0758-4ba0-9da4-3f5fdb7b0a68,43c0673b-610d-4540-863e-e501e7972044,7.609068,51.965790,7.610714,51.964908,90.0
1,38882f74-9d5b-47af-ba82-e93ad50d66cf,43c0673b-610d-4540-863e-e501e7972044,7.610714,51.964908,7.612946,51.964815,90.0


## Get the Segment Temperatures
The segment temperatures are stored in ``line_segment_res.csv`` as a long table with one
row per segment and timestamp. We load the data and parse the temperature values (the unit
`°C` is stripped off to get a numeric column):


In [4]:
# NBVAL_CHECK_OUTPUT
segment_res = pd.read_csv(os.path.join(result_path, "line_segment_res.csv"))
segment_res["lineSegmentTemperature_C"] = (
    segment_res["lineSegmentTemperature_C"].str.removesuffix("°C").astype(float)
)
# The timestamps are written with a timezone (UTC), strip the timezone information to allow matching
segment_res["time"] = (
    pd.to_datetime(segment_res["time"], utc=True)
    .dt.tz_convert("UTC")
    .dt.tz_localize(None)
)
segment_res.head()


,time,lineSegmentUuid,lineSegmentTemperature_C
0,2016-01-01 01:00:00,38882f74-9d5b-47af-ba82-e93ad50d66cf,30.790647
1,2016-01-01 01:00:00,f860a372-0758-4ba0-9da4-3f5fdb7b0a68,30.790647
2,2016-01-01 02:00:00,38882f74-9d5b-47af-ba82-e93ad50d66cf,43.126553
3,2016-01-01 02:00:00,f860a372-0758-4ba0-9da4-3f5fdb7b0a68,43.126553
4,2016-01-01 03:00:00,38882f74-9d5b-47af-ba82-e93ad50d66cf,51.910911


We want to plot the segment temperature for a specific timestamp, thus we need to filter
the data and map the temperature to the segment UUID:


In [5]:
# NBVAL_CHECK_OUTPUT
specific_time = pd.to_datetime("2016-01-01 12:00:00")
segment_temperatures = (
    segment_res.loc[segment_res["time"] == specific_time]
    .set_index("lineSegmentUuid")["lineSegmentTemperature_C"]
    .to_dict()
)
segment_temperatures


{'38882f74-9d5b-47af-ba82-e93ad50d66cf': 72.93007124561679,
 'f860a372-0758-4ba0-9da4-3f5fdb7b0a68': 72.93007124561679}

## Plot the Thermal Line Segments
With `thermal_line_segment_plot` from `pypsdm.plots.grid` the segments are drawn on top of
the grid. The color of each segment is determined by its value (here: temperature in °C)
using a colormap. You can choose any Plotly colormap, e.g. `Jet` or `Viridis`:


In [6]:
from pypsdm.plots.grid import thermal_line_segment_plot
thermal_line_segment_plot(
    gwr.grid,
    segments,
    segment_values=segment_temperatures,
    cmap="Jet",
    cbar_title="Segment Temperature [°C]",
    value_range=(0, 100),
)

If you want to see only the geometry of the segments (e.g. how the line was split),
just omit `segment_values`. Each segment then gets an individual color from a
qualitative palette so that the segments can be differentiated easily:

In [7]:
thermal_line_segment_plot(gwr.grid, segments)


You can also compare different timestamps by simply changing the filter time above.
As with the other grid plots you can adapt the base map via `mapbox_style`:


In [8]:
thermal_line_segment_plot(
    gwr.grid,
    segments,
    segment_values=segment_temperatures,
    cmap="Jet",
    cbar_title="Segment Temperature [°C]",
    mapbox_style="white-bg",
    value_range=(0, 100),
)


## Soil Layer Areas at a Given Depth
During the ampacity simulation the soil around the cables is modelled with soil
layers (``soilLayers.csv``) that reference a soil type (``soilTypes.csv``). Each
layer covers a depth range from ``z_from`` to ``z_to`` (in m, negative below the
surface).
To see which soil type is present at a specific depth (e.g. the cable burial
depth), the soil layer areas that cover that depth can be drawn on the map.
Each area gets an individual color so that the different areas can be
differentiated easily. The depth is entered by the user via `soil_depth` and
the transparency of the filled areas via `soil_opacity`:

In [9]:
# NBVAL_CHECK_OUTPUT
# Load the soil layers and soil types from the grid input data
soil_layers = pd.read_csv(os.path.join(grid_path, "soilLayers.csv"))
soil_types = pd.read_csv(os.path.join(grid_path, "soilTypes.csv"))
soil_layers

# Depth below the surface (in m, negative) at which the soil layer areas are shown
# e.g. the cable burial depth of -0.8 m
soil_depth = -1.0
# Transparency of the filled soil layer areas (0.0 = invisible, 1.0 = opaque)
soil_opacity = 0.5

thermal_line_segment_plot(
    gwr.grid,
    segments,
    segment_values=segment_temperatures,
    cmap="Jet",
    cbar_title="Segment Temperature [°C]",
    value_range=(0, 100),
    soil_layers=soil_layers,
    soil_types=soil_types,
    soil_depth=soil_depth,
    soil_opacity=soil_opacity,
)

## Segments and Soil Layers Only
To see the segment geometry together with the soil layer areas without any
segment values (and thus without the temperature colorbar), simply omit
`segment_values`. Each segment gets an individual color from a qualitative
palette and the soil layer areas are drawn in the background:

In [10]:
# NBVAL_CHECK_OUTPUT
thermal_line_segment_plot(
    gwr.grid,
    segments,
    soil_layers=soil_layers,
    soil_types=soil_types,
    soil_depth=soil_depth,
    soil_opacity=soil_opacity,
)
